# 🎓 Writing output in a Kernel

The typical way to write particle data to a file is by specifying the `output_file` argument in a `pset.execute()` call. This will write to a file on a regular interval, specified in the `outputdt` argument of the `ParticleFile`. 

However, sometimes you may want more control of _when_ to write particle data to a file. For example, you may want to write when a particle is deleted, or only when it is in a certain region or when other conditions are met. This can be especially useful if you want to do connectivity studies and don't care about having the full trajectory of each particle available. 

For these cases, you can use the `pfile.write()` method anywhere in a `Kernel`. You can write to the same file as the one specified in `output_file` or to a different file. 

This short tutorial will show you how to use `pfile.write()` in a kernel. We will use the same dataset and particle set as in the [output tutorial ](../../getting_started/tutorial_output.ipynb).


```{warning}
Writing output in a Kernel is an experimental feature and may change in the future.
```

In [ ]:
import numpy as np

import parcels
import parcels.tutorial

We start by defining two `ParticleFile` objects: one for writing at regular intervals and one for writing in the kernel. 

In [ ]:
# The file that will write at regular intervals (every 2 hours in this case)
output_file_regular = parcels.ParticleFile(
    "output_regular.parquet",
    outputdt=np.timedelta64(2, "h"),
)

# The file that will write when the `write_kernel` is executed.
output_file_kernel = parcels.ParticleFile(
    "output_kernel.parquet",
    outputdt=np.timedelta64(
        2, "h"
    ),  # TODO: this argument is not relevant for this file, since we will write to it manually in the kernel, but it still needs to be specified when creating the ParticleFile object
)

Now we define a kernel that writes to the file when executed. In this example, we write all particles to the file, but you can also choose to write only a subset of the particles by using indexing (e.g., `particles[0]` for the first particle). 

In [ ]:
def write_kernel(particles, fieldset):
    output_file_kernel.write(
        particles,
        particles.time,  # Note that here we use the time of the particles, which may not have started yet.
        fieldset=fieldset,  # Note that we need to specify the fieldset here as well, since the ParticleSetView does not have a reference to the fieldset.
    )

Now we run the simulation with both the advection kernel and the write kernel. Note that we specify the `output_file` argument in the `execute()` call, which will write to a file at regular intervals, but we can still use the `write_kernel` to write to a file at specific times.

In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)
ds_fields.load()  # load the dataset into memory

# Convert to SGRID-compliant dataset and create FieldSet
fields = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

pset = parcels.ParticleSet(fieldset=fieldset, lon=[31, 32], lat=[-32, -31], z=[1, 1])

pset.execute(
    kernels=[
        parcels.kernels.AdvectionRK2,
        write_kernel,  # the kernel that writes to the file when executed
    ],
    runtime=np.timedelta64(4, "h"),
    dt=np.timedelta64(15, "m"),
    output_file=output_file_regular,  # the file that writes at regular intervals
)

When we inspect the output files, we can see that the file that writes at regular intervals has 6 rows (2 particles * 3 time steps) at intervals of 2 hours, while the file that writes in the kernel has 32 rows (2 particles * 16 time steps) at intervals of 15 minutes.

In [ ]:
df_loop = parcels.read_particlefile(output_file_regular.path)
assert len(df_loop) == 6  # 2 particles * 3 time steps (0h, 2h, 4h)
df_loop

In [ ]:
output_file_kernel._writer.close()  # close the writer to flush the data to disk
df_kernel = parcels.read_particlefile(output_file_kernel.path)
assert (
    len(df_kernel) == 32
)  # 2 particles * 16 time steps (every 15 minutes for 4 hours)
df_kernel

## Writing on particle deletion

We can also write to the same file in both the `output_file` and the kernel. This could be useful if e.g. a particle is deleted in the kernel and you want to write the particle data at the time of deletion and you also want to have the full trajectory of the particle available in the output file.

In [ ]:
output_file_both = parcels.ParticleFile(
    "output_both.parquet",
    outputdt=np.timedelta64(2, "h"),
)


def write_kernel_on_delete(particles, fieldset):
    ptcls = particles[particles.time == 8100]  # delete particles at 2h15m
    ptcls.state = parcels.StatusCode.Delete
    if len(ptcls.time) > 0:
        output_file_both.write(
            ptcls,
            ptcls.time,
            fieldset=fieldset,
        )


pset = parcels.ParticleSet(fieldset=fieldset, lon=[31, 32], lat=[-32, -31], z=[1, 1])

pset.execute(
    kernels=[
        parcels.kernels.AdvectionRK2,
        write_kernel_on_delete,  # the kernel that writes to the file when executed
    ],
    runtime=np.timedelta64(4, "h"),
    dt=np.timedelta64(15, "m"),
    output_file=output_file_both,  # the file that writes at regular intervals
)

In [ ]:
df = parcels.read_particlefile(output_file_both.path)
assert len(df) == 6  # 2 particles * 3 time steps (0h, 2h, 2h15m)
df

As you can see in the output above, each particle is now written three times, at 0h, at 2h, and at 2h15m when the particle is deleted.

Of course, if you do not add the `output_file` argument to the `execute()` call, then only the kernel will write to the file and there will be one row per particle in the output file, at the time of deletion. Note that you then need to explicitly close the writer to flush the data to disk(with `output_file_both._writer.close()`), since the `execute()` call will not do this for you if you do not specify an `output_file`.